# Tool-RL (Phi-2) - Google Colab: SFT, RL, and eval

This notebook clones the project repo and, starting from `microsoft/phi-2`, evaluates the base model, runs SFT, runs GRPO + LoRA RL, and compares all three.
It runs the repo's own code, so push your changes before running it here (set `BRANCH` to test a branch).

In [ ]:
import os

REPO_URL = "https://github.com/codyho22/learn2call.git"
BRANCH = "main"
REPO_DIR = "/content/learn2call"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
%%writefile experiments/config.colab.phi2.yaml
run_name: colab_phi2
seed: 42

model:
  model_id: "microsoft/phi-2"
  trust_remote_code: true
  tokenizer_id: "microsoft/phi-2"

model_name: "microsoft/phi-2"
tokenizer_name: "microsoft/phi-2"

train:
  sft_epochs: 1
  per_device_batch_size: 1
  gradient_accumulation_steps: 8
  gradient_checkpointing: true
  rl_steps: 100
  learning_rate: 5.0e-5
  warmup_steps: 100
  max_length: 1024
  fp16: true
  bf16: false

rl:
  learning_rate: 1.0e-5
  per_device_batch_size: 4
  gradient_accumulation_steps: 2
  num_generations: 4
  max_completion_length: 128
  beta: 0.04
  lora_r: 16
  lora_alpha: 32

data:
  processed_path: "data/processed/train.jsonl"
  eval_path: "data/processed/val.jsonl"

logging:
  output_dir: "experiments/runs"
  log_interval: 20

In [ ]:
!python src/data.py --splits train --val-ratio 0.05 --seed 42

In [ ]:
!python experiments/test_setup.py

## Baseline: evaluate the base model
`EVAL_SAMPLES` val examples (of 1,029) are scored for each model; raise it for final numbers.

In [ ]:
EVAL_SAMPLES = 200
RUN_DIR = "experiments/runs/colab_phi2"
!mkdir -p {RUN_DIR}
!python src/eval.py --config experiments/config.colab.phi2.yaml --model microsoft/phi-2 --max-samples {EVAL_SAMPLES} --output {RUN_DIR}/preds_base.jsonl

## SFT

In [ ]:
!python src/train_sft.py --config experiments/config.colab.phi2.yaml --max-train-samples 1000 --max-eval-samples 100 --max-steps 200

In [ ]:
!ls -lah experiments/runs/colab_phi2/final

In [ ]:
!python src/eval.py --config experiments/config.colab.phi2.yaml --model {RUN_DIR}/final --max-samples {EVAL_SAMPLES} --output {RUN_DIR}/preds_sft.jsonl

## RL (GRPO + LoRA)
Starts from the SFT checkpoint above and saves a LoRA adapter to `experiments/runs/colab_phi2_rl/final`. Watch `reward` rise in the logs; the maximum is 3.0.

In [ ]:
!python src/train_rl.py --config experiments/config.colab.phi2.yaml

In [ ]:
!python src/eval.py --config experiments/config.colab.phi2.yaml --model {RUN_DIR}_rl/final --max-samples {EVAL_SAMPLES} --output {RUN_DIR}/preds_rl.jsonl

## Compare base vs SFT vs RL

In [ ]:
import json
import sys

import pandas as pd

sys.path.insert(0, "src")
from eval import score

results = {}
for name in ("base", "sft", "rl"):
    preds_path = f"{RUN_DIR}/preds_{name}.jsonl"
    if os.path.exists(preds_path):
        with open(preds_path, encoding="utf-8") as fp:
            results[name] = score([json.loads(line) for line in fp])
pd.DataFrame(results).round(3)